In [3]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
import matplotlib.pyplot as plt
from sklearn.metrics import silhouette_score
from sklearn.impute import SimpleImputer

# Load your original dataset
df = pd.read_csv("synthetic_transaction_data.csv")

In [6]:
# --------------------------
# 데이터 전처리 개선 (메모리 효율적)
# --------------------------
# 수치형/범주형 변수 식별
numeric_features = df.select_dtypes(include=["number"]).columns.tolist()
categorical_features = df.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

# 고유값이 많은 범주형 변수 확인 및 필터링
print("범주형 변수의 고유값 수:")
high_cardinality_features = []
low_cardinality_features = []

for col in categorical_features:
    n_unique = df[col].nunique()
    print(f"{col}: {n_unique}")
    
    if n_unique > 100:  # 고유값이 100개 이상인 경우
        high_cardinality_features.append(col)
    else:
        low_cardinality_features.append(col)

print(f"\n고유값이 많은 변수 ({len(high_cardinality_features)}개): {high_cardinality_features}")
print(f"고유값이 적은 변수 ({len(low_cardinality_features)}개): {low_cardinality_features}")

# 고유값이 많은 변수 처리 방법 선택
# 1. 제외하기
categorical_features = low_cardinality_features

# 또는 2. 다른 인코딩 방법 사용 (예: 라벨 인코딩)
from sklearn.preprocessing import LabelEncoder

# 레이블 인코딩 결과를 저장할 새 데이터프레임
df_encoded = df[numeric_features].copy()

# 고유값이 적은 변수는 원-핫 인코딩
for col in low_cardinality_features:
    # 원-핫 인코딩
    dummies = pd.get_dummies(df[col], prefix=col, drop_first=True)
    df_encoded = pd.concat([df_encoded, dummies], axis=1)

# 고유값이 많은 변수는 레이블 인코딩
for col in high_cardinality_features:
    le = LabelEncoder()
    df_encoded[f"{col}_encoded"] = le.fit_transform(df[col].astype(str))

# 결측치 처리
df_encoded = df_encoded.fillna(df_encoded.median())

# 스케일링
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
X = scaler.fit_transform(df_encoded)

# 텐서 변환 (샘플링이 필요할 수 있음)
if len(X) > 100000:  # 데이터가 너무 크면 샘플링
    sample_idx = np.random.choice(len(X), 100000, replace=False)
    X_sampled = X[sample_idx]
    X_tensor = torch.tensor(X_sampled, dtype=torch.float32)
else:
    X_tensor = torch.tensor(X, dtype=torch.float32)

범주형 변수의 고유값 수:
transaction_time: 573504
recipient: 121
voice_match: 2
DeviceInfo: 1787
region: 6
receiver_bank: 6
is_new_account_for_user: 2
is_new_device: 2
ip_address: 590489
vpn: 2
payment_method: 6
intent: 5
authentication: 6
app_version: 300
rooting: 2

고유값이 많은 변수 (5개): ['transaction_time', 'recipient', 'DeviceInfo', 'ip_address', 'app_version']
고유값이 적은 변수 (10개): ['voice_match', 'region', 'receiver_bank', 'is_new_account_for_user', 'is_new_device', 'vpn', 'payment_method', 'intent', 'authentication', 'rooting']


In [9]:
# -------------------------- #
# Isolation Forest 개선 #
# -------------------------- #

print("Isolation Forest 최적화 시작...")

# 최적 contamination 값 찾기
contamination_values = [0.01, 0.03, 0.05, 0.07, 0.1]
silhouette_scores = []

print(f"총 {len(contamination_values)}개의 contamination 값에 대해 테스트합니다.")

for i, contamination in enumerate(contamination_values, 1):
    print(f"[{i}/{len(contamination_values)}] contamination={contamination} 테스트 중...")
    
    model_if = IsolationForest(n_estimators=100, contamination=contamination, random_state=42)
    print("  - 모델 학습 중...")
    y_pred = model_if.fit_predict(X)
    y_pred = np.where(y_pred == 1, 0, 1)  # 이상치 = 1로 변환
    
    # 최소 2개 이상의 클러스터가 있고 각 클러스터에 최소 2개 이상의 샘플이 있어야 실루엣 점수 계산 가능
    if len(np.unique(y_pred)) > 1 and np.min(np.bincount(y_pred)) >= 2:
        print("  - 실루엣 점수 계산 중...")
        score = silhouette_score(X, y_pred)
        silhouette_scores.append(score)
        print(f"  - 실루엣 점수: {score:.4f}")
    else:
        silhouette_scores.append(-1)  # 유효하지 않은 클러스터링에 대한 페널티
        print("  - 유효하지 않은 클러스터링: 점수 -1 할당")

print("\n실루엣 점수 요약:")
for c, s in zip(contamination_values, silhouette_scores):
    print(f"contamination={c}: 실루엣 점수 {s:.4f}" if s != -1 else f"contamination={c}: 유효하지 않은 클러스터링")

best_contamination = contamination_values[np.argmax(silhouette_scores)]
best_score = max(silhouette_scores)
print(f"\n최적 contamination 값: {best_contamination} (실루엣 점수: {best_score:.4f})")

print("\n최종 모델 학습 및 이상치 검출 중...")
model_if = IsolationForest(n_estimators=100, contamination=best_contamination, random_state=42)
df["anomaly_if"] = model_if.fit_predict(X)
df["anomaly_if"] = df["anomaly_if"].map({1: 0, -1: 1})
df["score_if"] = model_if.decision_function(X) * -1  # 점수화

# 이상치 개수 확인
anomaly_count = df["anomaly_if"].sum()
total_count = len(df)
print(f"이상치 개수: {anomaly_count}/{total_count} ({anomaly_count/total_count*100:.2f}%)")
print("Isolation Forest 최적화 완료!")

Isolation Forest 최적화 시작...
총 5개의 contamination 값에 대해 테스트합니다.
[1/5] contamination=0.01 테스트 중...
  - 모델 학습 중...
  - 실루엣 점수 계산 중...
  - 실루엣 점수: 0.0350
[2/5] contamination=0.03 테스트 중...
  - 모델 학습 중...
  - 실루엣 점수 계산 중...
  - 실루엣 점수: 0.0354
[3/5] contamination=0.05 테스트 중...
  - 모델 학습 중...
  - 실루엣 점수 계산 중...
  - 실루엣 점수: 0.0351
[4/5] contamination=0.07 테스트 중...
  - 모델 학습 중...
  - 실루엣 점수 계산 중...
  - 실루엣 점수: 0.0343
[5/5] contamination=0.1 테스트 중...
  - 모델 학습 중...
  - 실루엣 점수 계산 중...
  - 실루엣 점수: 0.0329

실루엣 점수 요약:
contamination=0.01: 실루엣 점수 0.0350
contamination=0.03: 실루엣 점수 0.0354
contamination=0.05: 실루엣 점수 0.0351
contamination=0.07: 실루엣 점수 0.0343
contamination=0.1: 실루엣 점수 0.0329

최적 contamination 값: 0.03 (실루엣 점수: 0.0354)

최종 모델 학습 및 이상치 검출 중...
이상치 개수: 17717/590540 (3.00%)
Isolation Forest 최적화 완료!


In [11]:
# --------------------------
# AutoEncoder 개선
# --------------------------
print("AutoEncoder 모델 개선 작업 시작...")

# 데이터 준비 확인
print(f"데이터 형태 확인: X 크기 = {X.shape}, df 크기 = {df.shape}")

# 동일한 데이터셋 사용 보장
X_tensor = torch.FloatTensor(X)
print(f"텐서 변환 완료: X_tensor 크기 = {X_tensor.shape}")

# 아키텍처 개선
class ImprovedAutoEncoder(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        hidden_dim = max(16, input_dim // 4)  # 입력 차원에 비례한 은닉층 크기
        bottleneck_dim = max(8, input_dim // 8)  # 입력 차원에 비례한 병목층 크기
        
        print(f"모델 구성: 입력 차원 = {input_dim}, 은닉층 = {hidden_dim}, 병목층 = {bottleneck_dim}")
        
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, bottleneck_dim)
        )
        
        self.decoder = nn.Sequential(
            nn.Linear(bottleneck_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, input_dim),
            nn.Sigmoid()
        )
    
    def forward(self, x):
        return self.decoder(self.encoder(x))

# 배치 처리 및 조기 중단
batch_size = 64
print(f"배치 크기: {batch_size}")
train_dataset = TensorDataset(X_tensor)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
print(f"데이터 로더 준비 완료: {len(train_loader)} 배치")

ae = ImprovedAutoEncoder(X.shape[1])
optimizer = torch.optim.Adam(ae.parameters(), lr=1e-3, weight_decay=1e-5)
loss_fn = nn.MSELoss()
print("모델 및 옵티마이저 초기화 완료")

best_loss = float('inf')
patience = 5
patience_counter = 0
max_epochs = 100

print(f"학습 시작 - 최대 에폭: {max_epochs}, 조기 중단 인내: {patience}")
for epoch in range(max_epochs):
    ae.train()
    total_loss = 0
    batch_count = 0
    
    print(f"[에폭 {epoch+1}/{max_epochs}] 학습 진행 중...")
    for batch in train_loader:
        batch_count += 1
        if batch_count % 50 == 0:  # 50 배치마다 진행 상황 출력
            print(f"  - 배치 {batch_count}/{len(train_loader)} 처리 중...")
        
        x = batch[0]
        optimizer.zero_grad()
        output = ae(x)
        loss = loss_fn(output, x)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)
    
    avg_loss = total_loss / len(train_dataset)
    print(f"[에폭 {epoch+1}/{max_epochs}] 평균 손실: {avg_loss:.6f}")
    
    # 조기 중단 로직
    if avg_loss < best_loss:
        best_loss = avg_loss
        patience_counter = 0
        torch.save(ae.state_dict(), "best_ae_model.pt")
        print(f"  - 새로운 최고 모델 저장 (손실: {best_loss:.6f})")
    else:
        patience_counter += 1
        print(f"  - 개선 없음: {patience_counter}/{patience}")
        if patience_counter >= patience:
            print(f"  - 조기 중단! {epoch+1}번째 에폭에서 학습 종료")
            break

print("\n최고 모델 로드 중...")
ae.load_state_dict(torch.load("best_ae_model.pt"))

# 재구성 오차 계산 - df와 동일한 크기의 결과 보장
print("재구성 오차 계산 중...")
ae.eval()
recon_errors = []

# 배치 진행 상황 표시
total_batches = len(train_loader)
with torch.no_grad():
    for i, batch in enumerate(train_loader):
        if (i + 1) % 20 == 0 or (i + 1) == total_batches:
            print(f"  - 배치 {i+1}/{total_batches} 처리 중...")
        
        x = batch[0]
        output = ae(x)
        batch_errors = torch.mean((x - output) ** 2, dim=1).numpy()
        recon_errors.extend(batch_errors)

recon_error = np.array(recon_errors)
print(f"재구성 오차 계산 완료: {len(recon_error)} 샘플")

# 데이터 크기 확인 및 일치 보장
print(f"재구성 오차 배열 크기: {recon_error.shape}, DataFrame 크기: {df.shape}")
if len(recon_error) != len(df):
    print("경고: 오차 배열과 DataFrame 크기가 일치하지 않습니다!")
    print("X와 df의 데이터가 동일하지 않거나 일부 데이터만 처리되었을 수 있습니다.")
    print("DataFrame과 같은 크기로 재구성 오차 배열 조정...")
    
    # X와 df의 관계를 확인해야 합니다
    # 예시로 처음 행들에 대한 오차만 사용하거나, 필요한 만큼 복제하는 방식을 보여드립니다
    if len(recon_error) < len(df):
        # 부족한 경우 (실제로는 이렇게 해결하면 안 됨 - 단지 예시)
        recon_error = np.pad(recon_error, (0, len(df) - len(recon_error)), 'constant', constant_values=np.mean(recon_error))
    else:
        # 많은 경우
        recon_error = recon_error[:len(df)]
    
    print(f"조정된 재구성 오차 배열 크기: {recon_error.shape}")

# 최적 임계값 찾기 (통계적 방법)
mean_error = np.mean(recon_error)
std_error = np.std(recon_error)
threshold_ae = mean_error + 3 * std_error  # 3-시그마 규칙
print(f"임계값 계산: 평균 = {mean_error:.6f}, 표준편차 = {std_error:.6f}, 임계값 = {threshold_ae:.6f}")

print("이상치 탐지 결과 저장 중...")
df["anomaly_ae"] = (recon_error > threshold_ae).astype(int)
df["score_ae"] = recon_error  # 점수화

# 이상치 통계
anomaly_count = df["anomaly_ae"].sum()
print(f"탐지된 이상치: {anomaly_count}개 ({anomaly_count/len(df)*100:.2f}%)")
print("AutoEncoder 모델 개선 작업 완료!")

AutoEncoder 모델 개선 작업 시작...
데이터 형태 확인: X 크기 = (590540, 42), df 크기 = (590540, 25)
텐서 변환 완료: X_tensor 크기 = torch.Size([590540, 42])
배치 크기: 64
데이터 로더 준비 완료: 9228 배치
모델 구성: 입력 차원 = 42, 은닉층 = 16, 병목층 = 8
모델 및 옵티마이저 초기화 완료
학습 시작 - 최대 에폭: 100, 조기 중단 인내: 5
[에폭 1/100] 학습 진행 중...
  - 배치 50/9228 처리 중...
  - 배치 100/9228 처리 중...
  - 배치 150/9228 처리 중...
  - 배치 200/9228 처리 중...
  - 배치 250/9228 처리 중...
  - 배치 300/9228 처리 중...
  - 배치 350/9228 처리 중...
  - 배치 400/9228 처리 중...
  - 배치 450/9228 처리 중...
  - 배치 500/9228 처리 중...
  - 배치 550/9228 처리 중...
  - 배치 600/9228 처리 중...
  - 배치 650/9228 처리 중...
  - 배치 700/9228 처리 중...
  - 배치 750/9228 처리 중...
  - 배치 800/9228 처리 중...
  - 배치 850/9228 처리 중...
  - 배치 900/9228 처리 중...
  - 배치 950/9228 처리 중...
  - 배치 1000/9228 처리 중...
  - 배치 1050/9228 처리 중...
  - 배치 1100/9228 처리 중...
  - 배치 1150/9228 처리 중...
  - 배치 1200/9228 처리 중...
  - 배치 1250/9228 처리 중...
  - 배치 1300/9228 처리 중...
  - 배치 1350/9228 처리 중...
  - 배치 1400/9228 처리 중...
  - 배치 1450/9228 처리 중...
  - 배치 1500/9228 처리 중...


In [12]:
# --------------------------
# VAE 개선
# --------------------------
print("VAE 모델 개선 작업 시작...")

# 데이터 준비 확인
print(f"데이터 형태 확인: X 크기 = {X.shape}, df 크기 = {df.shape}")

# 동일한 데이터셋 사용 보장
X_tensor = torch.FloatTensor(X)
print(f"텐서 변환 완료: X_tensor 크기 = {X_tensor.shape}")

# VAE 아키텍처 개선
class ImprovedVAE(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        hidden_dim = max(16, input_dim // 4)  # 입력 차원에 비례한 은닉층 크기
        latent_dim = max(8, input_dim // 8)   # 입력 차원에 비례한 잠재 공간 차원
        
        print(f"VAE 모델 구성: 입력 차원 = {input_dim}, 은닉층 = {hidden_dim}, 잠재 차원 = {latent_dim}")
        
        # 인코더 네트워크
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2)
        )
        
        # 잠재 공간 매핑 (평균과 로그 분산)
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)
        
        # 디코더 네트워크
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, input_dim),
            nn.Sigmoid()
        )
        
        self.latent_dim = latent_dim
    
    def encode(self, x):
        h = self.encoder(x)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar
    
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        z = mu + eps * std
        return z
    
    def decode(self, z):
        return self.decoder(z)
    
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        recon_x = self.decode(z)
        return recon_x, mu, logvar

# 손실 함수 정의
def vae_loss_function(recon_x, x, mu, logvar, kld_weight=0.005):
    # 재구성 손실 (MSE)
    recon_loss = nn.MSELoss(reduction='sum')(recon_x, x)
    
    # KL 발산
    kld_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    
    # 전체 손실
    return recon_loss + kld_weight * kld_loss, recon_loss, kld_loss

# 배치 처리 및 조기 중단
batch_size = 64
print(f"배치 크기: {batch_size}")
train_dataset = TensorDataset(X_tensor)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
print(f"데이터 로더 준비 완료: {len(train_loader)} 배치")

vae = ImprovedVAE(X.shape[1])
optimizer = torch.optim.Adam(vae.parameters(), lr=1e-3, weight_decay=1e-5)
print("VAE 모델 및 옵티마이저 초기화 완료")

best_loss = float('inf')
patience = 5
patience_counter = 0
max_epochs = 100

print(f"학습 시작 - 최대 에폭: {max_epochs}, 조기 중단 인내: {patience}")
for epoch in range(max_epochs):
    vae.train()
    total_loss = 0
    recon_loss_sum = 0
    kld_loss_sum = 0
    batch_count = 0
    
    print(f"[에폭 {epoch+1}/{max_epochs}] 학습 진행 중...")
    for batch in train_loader:
        batch_count += 1
        if batch_count % 50 == 0:  # 50 배치마다 진행 상황 출력
            print(f"  - 배치 {batch_count}/{len(train_loader)} 처리 중...")
        
        x = batch[0]
        optimizer.zero_grad()
        
        recon_x, mu, logvar = vae(x)
        loss, recon_loss, kld_loss = vae_loss_function(recon_x, x, mu, logvar)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        recon_loss_sum += recon_loss.item()
        kld_loss_sum += kld_loss.item()
    
    avg_loss = total_loss / len(train_dataset)
    avg_recon_loss = recon_loss_sum / len(train_dataset)
    avg_kld_loss = kld_loss_sum / len(train_dataset)
    
    print(f"[에폭 {epoch+1}/{max_epochs}] 총 손실: {avg_loss:.6f}, 재구성 손실: {avg_recon_loss:.6f}, KLD 손실: {avg_kld_loss:.6f}")
    
    # 조기 중단 로직
    if avg_loss < best_loss:
        best_loss = avg_loss
        patience_counter = 0
        torch.save(vae.state_dict(), "best_vae_model.pt")
        print(f"  - 새로운 최고 모델 저장 (손실: {best_loss:.6f})")
    else:
        patience_counter += 1
        print(f"  - 개선 없음: {patience_counter}/{patience}")
        if patience_counter >= patience:
            print(f"  - 조기 중단! {epoch+1}번째 에폭에서 학습 종료")
            break

print("\n최고 모델 로드 중...")
vae.load_state_dict(torch.load("best_vae_model.pt"))

# 재구성 오차 계산 - df와 동일한 크기의 결과 보장
print("재구성 오차 계산 중...")
vae.eval()
recon_errors = []

# 배치 진행 상황 표시
total_batches = len(train_loader)
with torch.no_grad():
    for i, batch in enumerate(train_loader):
        if (i + 1) % 20 == 0 or (i + 1) == total_batches:
            print(f"  - 배치 {i+1}/{total_batches} 처리 중...")
        
        x = batch[0]
        recon_x, mu, logvar = vae(x)
        
        # 샘플별 MSE 계산
        batch_errors = torch.mean((x - recon_x) ** 2, dim=1).numpy()
        recon_errors.extend(batch_errors)

recon_error = np.array(recon_errors)
print(f"재구성 오차 계산 완료: {len(recon_error)} 샘플")

# 데이터 크기 확인 및 일치 보장
print(f"재구성 오차 배열 크기: {recon_error.shape}, DataFrame 크기: {df.shape}")
if len(recon_error) != len(df):
    print("경고: 오차 배열과 DataFrame 크기가 일치하지 않습니다!")
    print("X와 df의 데이터가 동일하지 않거나 일부 데이터만 처리되었을 수 있습니다.")
    print("DataFrame과 같은 크기로 재구성 오차 배열 조정...")
    
    if len(recon_error) < len(df):
        # 부족한 경우 (실제로는 이렇게 해결하면 안 됨 - 단지 예시)
        recon_error = np.pad(recon_error, (0, len(df) - len(recon_error)), 'constant', constant_values=np.mean(recon_error))
    else:
        # 많은 경우
        recon_error = recon_error[:len(df)]
    
    print(f"조정된 재구성 오차 배열 크기: {recon_error.shape}")

# 최적 임계값 찾기 (통계적 방법)
mean_error = np.mean(recon_error)
std_error = np.std(recon_error)
threshold_vae = mean_error + 3 * std_error  # 3-시그마 규칙
print(f"임계값 계산: 평균 = {mean_error:.6f}, 표준편차 = {std_error:.6f}, 임계값 = {threshold_vae:.6f}")

print("이상치 탐지 결과 저장 중...")
df["anomaly_vae"] = (recon_error > threshold_vae).astype(int)
df["score_vae"] = recon_error  # 점수화

# 이상치 통계
anomaly_count = df["anomaly_vae"].sum()
print(f"탐지된 이상치: {anomaly_count}개 ({anomaly_count/len(df)*100:.2f}%)")
print("VAE 모델 개선 작업 완료!")

VAE 모델 개선 작업 시작...
데이터 형태 확인: X 크기 = (590540, 42), df 크기 = (590540, 27)
텐서 변환 완료: X_tensor 크기 = torch.Size([590540, 42])
배치 크기: 64
데이터 로더 준비 완료: 9228 배치
VAE 모델 구성: 입력 차원 = 42, 은닉층 = 16, 잠재 차원 = 8
VAE 모델 및 옵티마이저 초기화 완료
학습 시작 - 최대 에폭: 100, 조기 중단 인내: 5
[에폭 1/100] 학습 진행 중...
  - 배치 50/9228 처리 중...
  - 배치 100/9228 처리 중...
  - 배치 150/9228 처리 중...
  - 배치 200/9228 처리 중...
  - 배치 250/9228 처리 중...
  - 배치 300/9228 처리 중...
  - 배치 350/9228 처리 중...
  - 배치 400/9228 처리 중...
  - 배치 450/9228 처리 중...
  - 배치 500/9228 처리 중...
  - 배치 550/9228 처리 중...
  - 배치 600/9228 처리 중...
  - 배치 650/9228 처리 중...
  - 배치 700/9228 처리 중...
  - 배치 750/9228 처리 중...
  - 배치 800/9228 처리 중...
  - 배치 850/9228 처리 중...
  - 배치 900/9228 처리 중...
  - 배치 950/9228 처리 중...
  - 배치 1000/9228 처리 중...
  - 배치 1050/9228 처리 중...
  - 배치 1100/9228 처리 중...
  - 배치 1150/9228 처리 중...
  - 배치 1200/9228 처리 중...
  - 배치 1250/9228 처리 중...
  - 배치 1300/9228 처리 중...
  - 배치 1350/9228 처리 중...
  - 배치 1400/9228 처리 중...
  - 배치 1450/9228 처리 중...
  - 배치 1500/9228 처리 중..

In [13]:
# --------------------------
# 모델 결과 저장
# --------------------------
print("\n모델 결과 저장 작업 시작...")

# 1. DataFrame 저장
print("이상치 탐지 결과가 포함된 DataFrame 저장 중...")
output_df_path = "df_unsupervised_after.csv"
df.to_csv(output_df_path, index=False)
print(f"DataFrame을 '{output_df_path}'에 저장했습니다.")

# 2. Isolation Forest 결과 저장
if "score_if" in df.columns:
    print("Isolation Forest 점수 저장 중...")
    if_scores_path = "if_scores.npy"
    np.save(if_scores_path, df["score_if"].values)
    print(f"Isolation Forest 점수를 '{if_scores_path}'에 저장했습니다.")
    
    # 예측 결과도 저장 (필요한 경우)
    if "anomaly_if" in df.columns:
        if_predictions_path = "if_predictions.npy"
        np.save(if_predictions_path, df["anomaly_if"].values)
        print(f"Isolation Forest 예측 결과를 '{if_predictions_path}'에 저장했습니다.")

# 3. 재구성 오차 저장 (AutoEncoder)
if "score_ae" in df.columns:
    print("AutoEncoder 재구성 오차 저장 중...")
    recon_error_path = "recon_error.npy"
    np.save(recon_error_path, df["score_ae"].values)
    print(f"AutoEncoder 재구성 오차를 '{recon_error_path}'에 저장했습니다.")

# 4. 재구성 오차 저장 (VAE)
if "score_vae" in df.columns:
    print("VAE 재구성 오차 저장 중...")
    recon_error_vae_path = "recon_error_vae.npy"
    np.save(recon_error_vae_path, df["score_vae"].values)
    print(f"VAE 재구성 오차를 '{recon_error_vae_path}'에 저장했습니다.")

# 5. 앙상블 및 개별 모델 점수 저장
print("모든 모델 점수를 딕셔너리 형태로 저장 중...")
anomaly_scores = {}

if "score_if" in df.columns:
    anomaly_scores["isolation_forest"] = df["score_if"].values
    print("- Isolation Forest 점수 추가")
    
if "score_ae" in df.columns:
    anomaly_scores["autoencoder"] = df["score_ae"].values
    print("- AutoEncoder 점수 추가")
    
if "score_vae" in df.columns:
    anomaly_scores["vae"] = df["score_vae"].values
    print("- VAE 점수 추가")
    
if "score_ensemble" in df.columns:
    anomaly_scores["ensemble"] = df["score_ensemble"].values
    print("- 앙상블 점수 추가")

anomaly_scores_path = "anomaly_scores.npy"
np.save(anomaly_scores_path, anomaly_scores)
print(f"모든 모델 점수를 '{anomaly_scores_path}'에 저장했습니다.")

# 6. 임계값 정보 저장 (추후 참조를 위해)
threshold_info = {}

if "threshold_ae" in locals() or "threshold_ae" in globals():
    threshold_info["autoencoder"] = threshold_ae
    print(f"- AutoEncoder 임계값 저장: {threshold_ae:.6f}")

if "threshold_vae" in locals() or "threshold_vae" in globals():
    threshold_info["vae"] = threshold_vae
    print(f"- VAE 임계값 저장: {threshold_vae:.6f}")

if "score_if" in df.columns:
    threshold_if = np.percentile(df["score_if"], 95)
    threshold_info["isolation_forest"] = threshold_if
    print(f"- Isolation Forest 임계값 저장: {threshold_if:.6f}")

if "score_ensemble" in df.columns:
    threshold_ensemble = np.percentile(df["score_ensemble"], 95)
    threshold_info["ensemble"] = threshold_ensemble
    print(f"- 앙상블 임계값 저장: {threshold_ensemble:.6f}")

threshold_path = "anomaly_thresholds.npy"
np.save(threshold_path, threshold_info)
print(f"모든 임계값 정보를 '{threshold_path}'에 저장했습니다.")

# 7. 모델 저장 (선택 사항)
print("\n모델 파일 저장 상태:")
model_files = {
    "AutoEncoder": "best_ae_model.pt",
    "VAE": "best_vae_model.pt"
}

for model_name, file_path in model_files.items():
    import os
    if os.path.exists(file_path):
        print(f"- {model_name} 모델이 '{file_path}'에 저장되어 있습니다.")
    else:
        print(f"- 경고: {model_name} 모델 파일을 찾을 수 없습니다.")

print("\n이상치 탐지 요약 정보:")
for col in ["anomaly_if", "anomaly_ae", "anomaly_vae", "anomaly_ensemble"]:
    if col in df.columns:
        count = df[col].sum()
        percent = count / len(df) * 100
        print(f"- {col}: {count}개 이상치 ({percent:.2f}%)")

print("모델 결과 저장 작업 완료!")
print(f"저장된 파일들은 다음 단계에서 다음과 같이 불러올 수 있습니다:")
print("- DataFrame: pd.read_csv('df_unsupervised_after.csv')")
print("- Isolation Forest 점수: np.load('if_scores.npy')")
print("- AutoEncoder 재구성 오차: np.load('recon_error.npy')")
print("- VAE 재구성 오차: np.load('recon_error_vae.npy')")
print("- 모든 모델 점수: np.load('anomaly_scores.npy', allow_pickle=True).item()")


모델 결과 저장 작업 시작...
이상치 탐지 결과가 포함된 DataFrame 저장 중...
DataFrame을 'df_unsupervised_after.csv'에 저장했습니다.
Isolation Forest 점수 저장 중...
Isolation Forest 점수를 'if_scores.npy'에 저장했습니다.
Isolation Forest 예측 결과를 'if_predictions.npy'에 저장했습니다.
AutoEncoder 재구성 오차 저장 중...
AutoEncoder 재구성 오차를 'recon_error.npy'에 저장했습니다.
VAE 재구성 오차 저장 중...
VAE 재구성 오차를 'recon_error_vae.npy'에 저장했습니다.
모든 모델 점수를 딕셔너리 형태로 저장 중...
- Isolation Forest 점수 추가
- AutoEncoder 점수 추가
- VAE 점수 추가
모든 모델 점수를 'anomaly_scores.npy'에 저장했습니다.
- AutoEncoder 임계값 저장: 0.121260
- VAE 임계값 저장: 0.122882
- Isolation Forest 임계값 저장: -0.005231
모든 임계값 정보를 'anomaly_thresholds.npy'에 저장했습니다.

모델 파일 저장 상태:
- AutoEncoder 모델이 'best_ae_model.pt'에 저장되어 있습니다.
- VAE 모델이 'best_vae_model.pt'에 저장되어 있습니다.

이상치 탐지 요약 정보:
- anomaly_if: 17717개 이상치 (3.00%)
- anomaly_ae: 3개 이상치 (0.00%)
- anomaly_vae: 97개 이상치 (0.02%)
모델 결과 저장 작업 완료!
저장된 파일들은 다음 단계에서 다음과 같이 불러올 수 있습니다:
- DataFrame: pd.read_csv('df_unsupervised_after.csv')
- Isolation Forest 점수: np.load('if_scores.npy')
- AutoEnc